In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

class ConnectX_NNUE(nn.Module):
    def __init__(self, input_dim=84):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        return torch.tanh(self.fc4(x))

class NNUE_Evaluator:
    """自动识别 84 / 85 维模型。
    85 维时启用 turn 身份特征；84 维时 turn 权重置零，行为与旧版完全一致。"""
    def __init__(self, model_path='nnue_model.pth'):
        sd = torch.load(model_path, map_location='cpu')
        self.input_dim = int(sd['fc1.weight'].shape[1])
        assert self.input_dim in (84, 85), f"不支持的输入维度 {self.input_dim}"
        model = ConnectX_NNUE(self.input_dim)
        model.load_state_dict(sd)
        model.eval()

        with torch.no_grad():
            w1 = np.ascontiguousarray(model.fc1.weight.numpy().T)      # (in_dim, 256)
            self.b1 = np.ascontiguousarray(model.fc1.bias.numpy())
            # 棋盘部分(前84行)用于累加器增量更新
            self.w1_board = np.ascontiguousarray(w1[:84])              # (84, 256)
            if self.input_dim == 85:
                self.w1_turn = np.ascontiguousarray(w1[84])           # (256,)
            else:
                self.w1_turn = np.zeros(256, dtype=np.float32)        # 84 维: turn 无贡献
            self.w1 = self.w1_board
            self.w2 = np.ascontiguousarray(model.fc2.weight.numpy().T)
            self.b2 = np.ascontiguousarray(model.fc2.bias.numpy())
            self.w3 = np.ascontiguousarray(model.fc3.weight.numpy().T)
            self.b3 = np.ascontiguousarray(model.fc3.bias.numpy())
            self.w4 = np.ascontiguousarray(model.fc4.weight.numpy().T)
            self.b4 = np.ascontiguousarray(model.fc4.bias.numpy())

        tag = ' , 启用 turn 特征' if self.input_dim == 85 else ''
        print(f"已加载模型 {model_path} (输入维度 {self.input_dim}{tag})")

    def get_initial_accumulator(self, board, player_mark):
        opp_mark = 2 if player_mark == 1 else 1
        features = np.zeros(84, dtype=np.float32)
        features[:42] = (board == player_mark).astype(np.float32)
        features[42:] = (board == opp_mark).astype(np.float32)
        return np.dot(features, self.w1_board) + self.b1


In [ ]:
import random
from numba import njit

# ===== Numba 加速的公共基础函数 =====

@njit(cache=True)
def check_win_numba(grid, piece):
    for r in range(6):
        for c in range(4):
            if grid[r][c]==piece and grid[r][c+1]==piece and grid[r][c+2]==piece and grid[r][c+3]==piece: return True
    for r in range(3):
        for c in range(7):
            if grid[r][c]==piece and grid[r+1][c]==piece and grid[r+2][c]==piece and grid[r+3][c]==piece: return True
    for r in range(3):
        for c in range(4):
            if grid[r][c]==piece and grid[r+1][c+1]==piece and grid[r+2][c+2]==piece and grid[r+3][c+3]==piece: return True
            if grid[r+3][c]==piece and grid[r+2][c+1]==piece and grid[r+1][c+2]==piece and grid[r][c+3]==piece: return True
    return False

@njit(cache=True)
def drop_piece_numba(grid, col, piece):
    for r in range(5, -1, -1):
        if grid[r][col] == 0:
            grid[r][col] = piece
            return r
    return -1


In [ ]:
import math

evaluator = None

# ===== NNUE 评估 + minimax (Numba 加速，支持 84/85 维) =====

# 已证明胜/负的分值基准：|值| >= WIN_SCORE 表示已证明的终局结果。
# NNUE 评估经 tanh，float32 下会饱和到恰好 ±1.0，因此必须与证明值严格分离。
WIN_SCORE = 100.0

@njit(cache=True, fastmath=True)
def evaluate_numba(acc, turn_flag, w1_turn, w2, b2, w3, b3, w4, b4, buf2, buf3):
    # 稀疏前向：跳过 ReLU 后为 0 的单元，避免小矩阵 BLAS 调用开销(约 40x 提速)
    # turn_flag: 1.0=轮到视角方(根玩家)落子, 0.0=轮到对方; 84维模型 w1_turn 全零 -> 无影响
    tf = np.float32(turn_flag)
    for j in range(64):
        buf2[j] = b2[j]
    for i in range(256):
        h = acc[i] + w1_turn[i] * tf
        if h > 0.0:
            for j in range(64):
                buf2[j] += h * w2[i, j]
    for j in range(32):
        buf3[j] = b3[j]
    for i in range(64):
        h = buf2[i]
        if h > 0.0:
            for j in range(32):
                buf3[j] += h * w3[i, j]
    s = b4[0]
    for i in range(32):
        h = buf3[i]
        if h > 0.0:
            s += h * w4[i, 0]
    return math.tanh(s)

@njit(cache=True)
def check_win_from_numba(grid, r, c):
    # 只检查经过 (r,c) 的四个方向是否连四，远快于全盘扫描
    piece = grid[r][c]
    # 横向
    cnt = 1
    cc = c - 1
    while cc >= 0 and grid[r][cc] == piece:
        cnt += 1; cc -= 1
    cc = c + 1
    while cc < 7 and grid[r][cc] == piece:
        cnt += 1; cc += 1
    if cnt >= 4: return True
    # 纵向(落子点在最上方，只需向下)
    cnt = 1
    rr = r + 1
    while rr < 6 and grid[rr][c] == piece:
        cnt += 1; rr += 1
    if cnt >= 4: return True
    # 对角线 反斜向
    cnt = 1
    rr = r - 1; cc = c - 1
    while rr >= 0 and cc >= 0 and grid[rr][cc] == piece:
        cnt += 1; rr -= 1; cc -= 1
    rr = r + 1; cc = c + 1
    while rr < 6 and cc < 7 and grid[rr][cc] == piece:
        cnt += 1; rr += 1; cc += 1
    if cnt >= 4: return True
    # 对角线 斜向
    cnt = 1
    rr = r - 1; cc = c + 1
    while rr >= 0 and cc < 7 and grid[rr][cc] == piece:
        cnt += 1; rr -= 1; cc += 1
    rr = r + 1; cc = c - 1
    while rr < 6 and cc >= 0 and grid[rr][cc] == piece:
        cnt += 1; rr += 1; cc -= 1
    return cnt >= 4

@njit(cache=True)
def minimax_nnue_numba(grid, depth, alpha, beta, is_max, my_piece, acc_stack, ply,
                       w1, w1_turn, w2, b2, w3, b3, w4, b4, buf2, buf3, pref):
    # 要求调用方保证当前局面非终局。grid 就地落子/撤销；acc_stack[ply] 为当前累加器，
    # 子结点写入 acc_stack[ply+1]，搜索全程零堆分配。
    # pref: 优先搜索的列(迭代加深传入上一层最优列)，-1 表示无。
    cols = np.empty(7, dtype=np.int32)
    rows = np.empty(7, dtype=np.int32)
    n = 0
    if pref >= 0 and grid[0][pref] == 0:
        cols[n] = pref; n += 1
    for c in (3, 2, 4, 1, 5, 0, 6):
        if c != pref and grid[0][c] == 0:
            cols[n] = c; n += 1
    if n == 0:
        return -1, 0.0  # 棋盘已满 = 平局
    opp = 2 if my_piece == 1 else 1
    mover = my_piece if is_max else opp
    # 即胜预检：行棋方存在立即连四 -> 直接返回证明值，无需展开子树
    for i in range(n):
        c = cols[i]
        r = 5
        while grid[r][c] != 0:
            r -= 1
        rows[i] = r
        grid[r][c] = mover
        won = check_win_from_numba(grid, r, c)
        grid[r][c] = 0
        if won:
            if is_max:
                return c, 100.0 + depth - 1.0
            return c, -(100.0 + depth - 1.0)
    best_col = cols[0]
    base = 0 if is_max else 42
    if is_max:
        v = -np.inf
        for i in range(n):
            c = cols[i]; r = rows[i]
            grid[r][c] = mover
            idx = base + r * 7 + c
            for k in range(256):
                acc_stack[ply + 1, k] = acc_stack[ply, k] + w1[idx, k]
            full = False
            if r == 0:
                full = True
                for cc in range(7):
                    if grid[0][cc] == 0:
                        full = False; break
            if full:
                score = 0.0
            elif depth <= 1:
                score = evaluate_numba(acc_stack[ply + 1], 0.0, w1_turn, w2, b2, w3, b3, w4, b4, buf2, buf3)
            else:
                _, score = minimax_nnue_numba(grid, depth - 1, alpha, beta, False, my_piece, acc_stack, ply + 1,
                                              w1, w1_turn, w2, b2, w3, b3, w4, b4, buf2, buf3, -1)
            grid[r][c] = 0
            if score > v:
                v = score; best_col = c
            if v > alpha: alpha = v
            if alpha >= beta: break
        return best_col, v
    else:
        v = np.inf
        for i in range(n):
            c = cols[i]; r = rows[i]
            grid[r][c] = mover
            idx = base + r * 7 + c
            for k in range(256):
                acc_stack[ply + 1, k] = acc_stack[ply, k] + w1[idx, k]
            full = False
            if r == 0:
                full = True
                for cc in range(7):
                    if grid[0][cc] == 0:
                        full = False; break
            if full:
                score = 0.0
            elif depth <= 1:
                score = evaluate_numba(acc_stack[ply + 1], 1.0, w1_turn, w2, b2, w3, b3, w4, b4, buf2, buf3)
            else:
                _, score = minimax_nnue_numba(grid, depth - 1, alpha, beta, True, my_piece, acc_stack, ply + 1,
                                              w1, w1_turn, w2, b2, w3, b3, w4, b4, buf2, buf3, -1)
            grid[r][c] = 0
            if score < v:
                v = score; best_col = c
            if v < beta: beta = v
            if alpha >= beta: break
        return best_col, v

def make_search_buffers(acc0):
    """为一次搜索准备 acc 栈与前向缓冲。acc_stack[0] = 根局面累加器。"""
    acc_stack = np.zeros((43, 256), dtype=np.float32)
    acc_stack[0] = acc0
    buf2 = np.empty(64, dtype=np.float32)
    buf3 = np.empty(32, dtype=np.float32)
    return acc_stack, buf2, buf3


In [ ]:
# ============================================================
# Cell 4: 配置区 —— 在此设置“测试模式 + 测试配置”，执行代码见下方各单元
# ============================================================
# MODE 可选：
#   'nnue_vs_nnue' —— 两个 NNUE 模型对战（固定深度）
#   'match'        —— 比赛模式：两对象对抗，不限深度、限制每步思考时间（迭代加深）
MODE = 'match'

# ==================== 固定深度模式参数 ====================
# ---- 模型路径 ----
MODEL_A_PATH = '/kaggle/input/connectx-nnue/nnue_model_a.pth'
MODEL_B_PATH = '/kaggle/input/connectx-nnue/nnue_model_b.pth'
# ---- 搜索深度 ----
DEPTH_A = 10           # nnue_vs_nnue：模型 A 深度
DEPTH_B = 10           # nnue_vs_nnue：模型 B 深度
# ---- 对战参数 ----
TOTAL_PAIRS   = 120    # 对战对数（每对打 2 局，共 TOTAL_PAIRS*2 局）
OPENING_STEPS = 4      # 随机开局步数（0 = 从空棋盘开始）

# ==================== 比赛模式(match) 参数 ====================
# 两个比赛对象，每个为：
#   {'type': 'nnue', 'path': '<模型.pth>', 'name': '显示名'}
MATCH_A = {'type': 'nnue', 'name': 'A',
           'path': '/kaggle/input/connectx-nnue/nnue_model_a.pth'}
MATCH_B = {'type': 'nnue', 'name': 'B',
           'path': '/kaggle/input/connectx-nnue/nnue_model_b.pth'}
MATCH_TIME_LIMIT    = 2.0   # 每步思考时间上限(秒)
MATCH_ROUNDS        = 2     # 对抗局数（默认 1 局，可设 >1 进行多轮对抗）
MATCH_SWAP          = True  # 多局时是否交替先后手(保证公平)
# 开局设置：
#   []        —— 从空棋盘开始
#   [0,2,6,2] —— 固定开局序列（按索引0~6指定落子列，双方交替落子）
#   整数 N    —— 随机生成 N 步开局
MATCH_OPENING       = []    # ← 在此修改开局设置
MATCH_MAX_DEPTH     = 42    # 迭代加深深度安全上限

# ==================== 打印当前配置 ====================
print(f'模式: {MODE}')
if MODE == 'nnue_vs_nnue':
    print(f'  模型 A : {MODEL_A_PATH}')
    print(f'  模型 B : {MODEL_B_PATH}')
    print(f'  深度 A : {DEPTH_A} | 深度 B: {DEPTH_B}')
    print(f'  对战: {TOTAL_PAIRS} 对 ({TOTAL_PAIRS*2} 局), 开局步数={OPENING_STEPS}')
elif MODE == 'match':
    print('  比赛模式(时间限制)：')
    print(f"    对象 {MATCH_A['name']}: NNUE({MATCH_A['path'].split('/')[-1]})")
    print(f"    对象 {MATCH_B['name']}: NNUE({MATCH_B['path'].split('/')[-1]})")
    _op_desc = MATCH_OPENING if isinstance(MATCH_OPENING, (list, tuple)) else f'随机{MATCH_OPENING}步'
    print(f'    每步 {MATCH_TIME_LIMIT}s | {MATCH_ROUNDS} 局 | 交替先后手={MATCH_SWAP} | 开局={_op_desc}')
else:
    raise ValueError(f"未知 MODE: {MODE!r}，请设为 'nnue_vs_nnue' / 'match'")


In [ ]:
# ============================================================
# Cell 5: 加载模型 + Numba 预热（按 MODE 自动加载所需 evaluator）
# ============================================================
evaluator = evaluator_a = evaluator_b = None
match_ev_a = match_ev_b = None

if MODE == 'nnue_vs_nnue':
    print('加载模型 A ...')
    evaluator_a = NNUE_Evaluator(MODEL_A_PATH)
    print('加载模型 B ...')
    evaluator_b = NNUE_Evaluator(MODEL_B_PATH)
    evaluator = evaluator_a   # 兼容单模型预热接口
elif MODE == 'match':
    if MATCH_A['type'] == 'nnue':
        print('加载比赛对象 A ...')
        match_ev_a = NNUE_Evaluator(MATCH_A['path'])
    if MATCH_B['type'] == 'nnue':
        print('加载比赛对象 B ...')
        match_ev_b = NNUE_Evaluator(MATCH_B['path'])
else:
    raise ValueError(f"未知 MODE: {MODE!r}")

# Numba 预热（只编译一次）
_wg  = np.zeros((6, 7), dtype=np.int32)
_acc = np.zeros(256, dtype=np.float32)
_ = check_win_numba(_wg, 1)
_ = drop_piece_numba(_wg.copy(), 0, 1)

# 收集本次需要预热的 NNUE evaluator（去重）
_seen, _warm = set(), []
for _e in (evaluator, evaluator_a, evaluator_b, match_ev_a, match_ev_b):
    if _e is not None and id(_e) not in _seen:
        _seen.add(id(_e)); _warm.append(_e)
for _ev in _warm:
    _stk, _b2, _b3 = make_search_buffers(_acc)
    _ = check_win_from_numba(_wg, 5, 0)
    _ = evaluate_numba(_acc, 1.0, _ev.w1_turn,
                       _ev.w2, _ev.b2, _ev.w3, _ev.b3, _ev.w4, _ev.b4, _b2, _b3)
    _ = minimax_nnue_numba(_wg, 2, -np.inf, np.inf, True, 1, _stk, 0,
                           _ev.w1, _ev.w1_turn,
                           _ev.w2, _ev.b2, _ev.w3, _ev.b3, _ev.w4, _ev.b4, _b2, _b3, -1)

print('Numba 预热完成。请运行下方与 MODE 对应的执行单元。')


In [ ]:
# ============================================================
# Cell 6: 执行对战（固定深度模式：nnue_vs_nnue）
# ============================================================
import time, random
from kaggle_environments import make

# ---------- 通用工具（比赛模式也会复用） ----------
def reward_to_str(r):
    return '胜' if r == 1 else ('平' if r == 0 else '负')

def gen_opening(steps):
    """生成 steps 步合法随机开局，返回落子列列表。"""
    if steps <= 0:
        return []
    opening = []
    tmp_grid = np.zeros((6, 7), dtype=np.int32)
    for step in range(steps):
        valid = [c for c in range(7) if tmp_grid[0][c] == 0]
        if not valid:
            break
        col = random.choice(valid)
        opening.append(col)
        drop_piece_numba(tmp_grid, col, 1 if step % 2 == 0 else 2)
    return opening

def make_nnue_agent(ev, depth):
    """返回绑定了指定 evaluator 和深度的独立 NNUE agent 闭包。"""
    def agent(obs, config):
        try:
            board = np.array(obs.board, dtype=np.int32)
            pieces = int(np.count_nonzero(board))
            if 'CURRENT_OPENING' in globals() and pieces < len(CURRENT_OPENING):
                return int(CURRENT_OPENING[pieces])
            grid = board.reshape(6, 7)
            acc_stack, buf2, buf3 = make_search_buffers(ev.get_initial_accumulator(board, obs.mark))
            col, _ = minimax_nnue_numba(
                grid, depth, -np.inf, np.inf, True, obs.mark, acc_stack, 0,
                ev.w1, ev.w1_turn, ev.w2, ev.b2, ev.w3, ev.b3, ev.w4, ev.b4, buf2, buf3, -1)
            return int(col)
        except Exception:
            return int(random.choice([c for c in range(7) if obs.board[c] == 0]))
    return agent

if MODE != 'nnue_vs_nnue':
    print(f"当前 MODE={MODE!r}，跳过固定深度对战；请运行下方“比赛模式”单元。")
else:
    env = make('connectx', debug=False)
    global CURRENT_OPENING
    score_a = 0.0
    start = time.time()

    # ========== NNUE A vs NNUE B ==========
    if MODE == 'nnue_vs_nnue':
        agent_a = make_nnue_agent(evaluator_a, DEPTH_A)
        agent_b = make_nnue_agent(evaluator_b, DEPTH_B)

        name_a = MODEL_A_PATH.split('/')[-1]
        name_b = MODEL_B_PATH.split('/')[-1]
        print(f'\n对战模式: 模型A [{name_a}]  vs  模型B [{name_b}]')
        print(f'深度: A={DEPTH_A}, B={DEPTH_B} | 共 {TOTAL_PAIRS} 对 ({TOTAL_PAIRS*2} 局), 开局步数={OPENING_STEPS}')
        print('=' * 70)

        for i in range(TOTAL_PAIRS):
            CURRENT_OPENING = gen_opening(OPENING_STEPS)

            env.reset()
            r1 = env.run([agent_a, agent_b])[-1][0]['reward']   # A 先手
            env.reset()
            r2 = env.run([agent_b, agent_a])[-1][1]['reward']   # A 后手

            ps = (1 if r1 == 1 else 0.5 if r1 == 0 else 0) + \
                 (1 if r2 == 1 else 0.5 if r2 == 0 else 0)
            score_a += ps
            print(f'Pair {i+1:2d} (开局 {CURRENT_OPENING}): '
                  f'A先手[{reward_to_str(r1)}], A后手[{reward_to_str(r2)}] | '
                  f'本轮: {ps}  累计: {score_a}')

    # ========== 汇总 ==========
    elapsed = time.time() - start
    total_games = TOTAL_PAIRS * 2
    score_b = total_games - score_a

    print('=' * 70)
    print(f'\n【汇总】共 {total_games} 局')
    print(f'  模型 A 得分: {score_a:.1f}  胜率: {score_a/total_games*100:.1f}%')
    print(f'  模型 B 得分: {score_b:.1f}  胜率: {score_b/total_games*100:.1f}%')
    winner = '模型 A' if score_a > score_b else ('模型 B' if score_b > score_a else '平局')
    print(f'  结果: {winner} 胜出' if winner != '平局' else '  结果: 平局')
    print(f'总耗时 {elapsed:.1f}s，平均每局 {elapsed/total_games:.2f}s')


In [ ]:
# ============================================================
# Cell 7: 执行对战（比赛模式：限制每步思考时间 + 迭代加深）
# ============================================================
if MODE != 'match':
    print(f"当前 MODE={MODE!r}，跳过比赛模式；请运行上方固定深度对战单元。")
else:
    import time, random
    from kaggle_environments import make

    # ---------- 构建“时间限制 + 迭代加深”的对战智能体 ----------
    def build_match_agent(spec, ev):
        """spec: 配置字典；ev: 已加载的 NNUE_Evaluator。
        使用迭代加深：depth=1,2,3,... 逐层加深，直到超过 MATCH_TIME_LIMIT
        或搜到必胜/必败，返回“最后一个完成深度”的结果，并打印每步日志。"""
        label = spec['name']
        if spec['type'] == 'nnue':
            tag = spec['path'].split('/')[-1]
            disp = f'NNUE({tag})'

            def agent(obs, config):
                board = np.array(obs.board, dtype=np.int32)
                pieces = int(np.count_nonzero(board))
                if 'CURRENT_OPENING' in globals() and pieces < len(CURRENT_OPENING):
                    mv = int(CURRENT_OPENING[pieces])
                    print(f'    [{label}|{disp}] mark={obs.mark} 开局手 落子={mv}')
                    return mv
                grid = board.reshape(6, 7)
                acc_stack, buf2, buf3 = make_search_buffers(ev.get_initial_accumulator(board, obs.mark))
                t0 = time.time()
                best_col, best_val, reached = -1, 0.0, 0
                depth = 1
                while depth <= MATCH_MAX_DEPTH:
                    col, val = minimax_nnue_numba(
                        grid, depth, -np.inf, np.inf, True, obs.mark, acc_stack, 0,
                        ev.w1, ev.w1_turn, ev.w2, ev.b2, ev.w3, ev.b3, ev.w4, ev.b4, buf2, buf3, best_col)
                    best_col, best_val, reached = int(col), float(val), depth
                    # 仅当 |val| >= WIN_SCORE（已证明的胜/负，而非 tanh 饱和到 ±1.0 的评估值）才提前停止
                    if time.time() - t0 >= MATCH_TIME_LIMIT or abs(val) >= WIN_SCORE:
                        break
                    depth += 1
                used = time.time() - t0
                if best_col < 0:
                    best_col = int(random.choice([c for c in range(7) if obs.board[c] == 0]))
                print(f'    [{label}|{disp}] mark={obs.mark} 落子={best_col} '
                      f'深度={reached} 评分={best_val:+.4f} 用时={used:.2f}s')
                return best_col
            return agent, disp

        raise ValueError(f"未知对象类型: {spec['type']!r}，应为 'nnue'")

    # ---------- 准备对战（复用 Cell 5 已加载的 evaluator） ----------
    agent_a, disp_a = build_match_agent(MATCH_A, match_ev_a)
    agent_b, disp_b = build_match_agent(MATCH_B, match_ev_b)

    # 放宽 env 的每步/整局超时，避免思考时间被环境判负（本地评测，非线上限制）
    _big = max(60.0, MATCH_TIME_LIMIT * 20 + 30)
    env = make('connectx', configuration={'actTimeout': _big, 'agentTimeout': _big,
                                           'runTimeout': _big * 100}, debug=True)

    def resolve_opening():
        """根据 MATCH_OPENING 类型解析开局：列表/元组直接用，整数随机生成。"""
        if isinstance(MATCH_OPENING, (list, tuple)):
            # 校验：确保指定的列是合法的（0-6 范围）
            opening = [int(c) for c in MATCH_OPENING]
            tmp_grid = np.zeros((6, 7), dtype=np.int32)
            valid_opening = []
            for step, col in enumerate(opening):
                if col < 0 or col > 6 or tmp_grid[0][col] != 0:
                    print(f'  警告: 开局第 {step+1} 步列 {col} 无效，截断开局序列。')
                    break
                valid_opening.append(col)
                drop_piece_numba(tmp_grid, col, 1 if step % 2 == 0 else 2)
            return valid_opening
        else:
            # 整数：随机生成 N 步
            return gen_opening(int(MATCH_OPENING))

    global CURRENT_OPENING
    wins_a = wins_b = draws = 0
    match_start = time.time()

    _op_desc = list(MATCH_OPENING) if isinstance(MATCH_OPENING, (list, tuple)) else f'随机{MATCH_OPENING}步'
    print('\n' + '=' * 70)
    print(f'比赛模式(时间限制) | A[{disp_a}]  vs  B[{disp_b}]')
    print(f'每步 {MATCH_TIME_LIMIT}s | {MATCH_ROUNDS} 局 | 交替先后手={MATCH_SWAP} | 开局={_op_desc}')
    print('=' * 70)

    for rnd in range(1, MATCH_ROUNDS + 1):
        CURRENT_OPENING = resolve_opening()
        a_first = (not MATCH_SWAP) or (rnd % 2 == 1)
        first_name = f'A[{disp_a}]' if a_first else f'B[{disp_b}]'
        print(f'\n----- 第 {rnd} 局 | 先手: {first_name} | 开局={CURRENT_OPENING} -----')

        env.reset()
        if a_first:
            res = env.run([agent_a, agent_b])
            rA = res[-1][0]['reward']
        else:
            res = env.run([agent_b, agent_a])
            rA = res[-1][1]['reward']

        if rA == 1:
            wins_a += 1; outcome = f'A[{disp_a}] 胜'
        elif rA == 0:
            draws += 1; outcome = '平局'
        else:
            wins_b += 1; outcome = f'B[{disp_b}] 胜'
        print(f'  => 结果: {outcome}  (A reward={rA})')

    # ---------- 汇总 ----------
    elapsed = time.time() - match_start
    print('\n' + '=' * 70)
    print(f'【比赛汇总】共 {MATCH_ROUNDS} 局')
    print(f'  A[{disp_a}] 胜: {wins_a}')
    print(f'  B[{disp_b}] 胜: {wins_b}')
    print(f'  平局: {draws}')
    if wins_a > wins_b:
        print(f'  最终结果: A[{disp_a}] 胜出')
    elif wins_b > wins_a:
        print(f'  最终结果: B[{disp_b}] 胜出')
    else:
        print('  最终结果: 平手')
    print(f'总耗时 {elapsed:.1f}s，平均每局 {elapsed / max(1, MATCH_ROUNDS):.2f}s')